In [1]:
%%time

import tensorflow as tf
print(tf.__version__)
import sys
sys.path.append("..")
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np


def plot_loss(history, *losses):
    for loss in losses:
        plt.plot(history.history[loss], label=loss)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

def scaling(x, min, max):
    return np.where(x < min, 0.0, np.where(x > max, 1.0, (x - min) / (max - min)))

early_stopping = EarlyStopping(
    monitor='val_loss',  # 
    patience=50,        # 
    verbose=1,          # 
    mode='min',         # 
    restore_best_weights=True  # 
)
from keras.callbacks import Callback

2024-01-26 10:22:33.549065: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-01-26 10:22:33.617960: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-01-26 10:22:33.618005: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-01-26 10:22:33.618072: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-01-26 10:22:33.634848: I tensorflow/core/platform/cpu_feature_g

2.14.0
CPU times: user 3.88 s, sys: 2.51 s, total: 6.39 s
Wall time: 3.58 s


In [2]:
SAVE_DIR = "../data"
file_criteo = SAVE_DIR + "/criteo-uplift-v2.1.csv"
df_criteo_ori = pd.read_csv(file_criteo, sep=',')

In [5]:
%%time
model_name = "TPM-XL"
result_list = []
DRP_aucc_test_list = []
roi_rank_pre_test_list = []
for sample in np.arange(0.11, 0.22, 0.01):
    print("sample_value = ", sample)
    random_state=20220720
    df_criteo=df_criteo_ori.sample(frac=sample, random_state=random_state).reset_index(drop=True)
    X = df_criteo[['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11']].values

    X[:, 0] = scaling(X[:, 0], min=np.min(X[:, 0]), max=np.max(X[:, 0]))
    X[:, 1] = scaling(X[:, 1], min=np.min(X[:, 1]), max=np.max(X[:, 1]))
    X[:, 2] = scaling(X[:, 2], min=np.min(X[:, 2]), max=np.max(X[:, 2]))
    X[:, 3] = scaling(X[:, 3], min=np.min(X[:, 3]), max=np.max(X[:, 3]))
    X[:, 4] = scaling(X[:, 4], min=np.min(X[:, 4]), max=np.max(X[:, 4]))
    X[:, 5] = scaling(X[:, 5], min=np.min(X[:, 5]), max=np.max(X[:, 5]))
    X[:, 6] = scaling(X[:, 6], min=np.min(X[:, 6]), max=np.max(X[:, 6]))
    X[:, 7] = scaling(X[:, 7], min=np.min(X[:, 7]), max=np.max(X[:, 7]))
    X[:, 8] = scaling(X[:, 8], min=np.min(X[:, 8]), max=np.max(X[:, 8]))
    X[:, 9] = scaling(X[:, 9], min=np.min(X[:, 9]), max=np.max(X[:, 9]))
    X[:, 10] = scaling(X[:, 10], min=np.min(X[:, 10]), max=np.max(X[:, 10]))
    X[:, 11] = scaling(X[:, 11], min=np.min(X[:, 11]), max=np.max(X[:, 11]))

    T = df_criteo['treatment'].values.reshape(-1, 1)
    Y_visit = df_criteo['visit'].values.reshape(-1, 1)
    Y_conv = df_criteo['conversion'].values.reshape(-1, 1)

    T.shape, Y_visit.shape, Y_conv.shape

    # calculate len
    train_len = int(len(X) * 0.70)
    cali_len = int(len(X) * 0.0)
    test_len = len(X) - train_len - cali_len

    # obtain train set
    X_train = X[:train_len, :]
    T_train = T[:train_len, :]
    Y_visit_train = Y_visit[:train_len, :]
    Y_conv_train = Y_conv[:train_len, :]

    # obtain calibration set
    X_cali = X[train_len:train_len+cali_len, :]
    T_cali = T[train_len:train_len+cali_len, :]
    Y_visit_cali = Y_visit[train_len:train_len+cali_len, :]
    Y_conv_cali = Y_conv[train_len:train_len+cali_len, :]

    # obtain test set
    X_test = X[train_len+cali_len:, :]
    T_test = T[train_len+cali_len:, :]
    Y_visit_test = Y_visit[train_len+cali_len:, :]
    Y_conv_test = Y_conv[train_len+cali_len:, :]

    #print(train_len, X_train.shape, X_test.shape, len(X), X_cali.shape, T_train.shape, Y_visit_train.shape)
    
    
    sys.path.append("..")
    from model.uplift_model import *

    count = 1
    # 2.5e-5
    import keras
    import keras.backend as K
    import tensorflow as tf
    from keras.callbacks import LearningRateScheduler
    from keras.callbacks import ReduceLROnPlateau
    from model.roi_model import *
    
    # visit
    final_model = get_slearner_criteo_model()
    final_model.load_weights('../model_file/uplift/criteo/final_model/slearner/c_slearner_criteo_model_1.h5')
    T0 = np.zeros(shape=T_train.shape)
    T1 = np.ones(shape=T_train.shape)
    p0 = final_model.predict([X_train,  T0])
    p1 = final_model.predict([X_train,  T1])
    
    X_train_0 = X_train[(T_train.flatten() < 0.5), :]
    X_train_1 = X_train[(T_train.flatten() >= 0.5), :]

    X_train_0_p1_pre = p1[(T_train < 0.5)]
    X_train_1_p0_pre = p0[(T_train >= 0.5)]

    Y_visit_train_0 = Y_visit_train[(T_train < 0.5)]
    Y_visit_train_1 = Y_visit_train[(T_train >= 0.5)]

    tau_0_label = X_train_0_p1_pre - Y_visit_train_0
    tau_1_label = Y_visit_train_1 - X_train_1_p0_pre
    
    
    final_model = get_xlearner_criteo_model()
    final_model.compile(loss='mse', optimizer='adam', metrics=['mse'])

    mcp_save = ModelCheckpoint('../model_file/uplift/criteo/final_model/xlearner/c_visit_xlearner_criteo_model_tau_0_{}.h5'.format(sample), save_best_only=True, monitor='val_loss', mode='min', save_weights_only=True)
    history = final_model.fit(X_train_0, tau_0_label, validation_split=0.2, epochs=500, batch_size=32688, shuffle=True, verbose=0, callbacks=[mcp_save])

    #plot_loss(history, "loss", "val_loss", "mse", "val_mse")
    
    
    final_model = get_xlearner_criteo_model()
    final_model.compile(loss='mse', optimizer='adam', metrics=['mse'])

    mcp_save = ModelCheckpoint('../model_file/uplift/criteo/final_model/xlearner/c_visit_xlearner_criteo_model_tau_1_{}.h5'.format(sample), save_best_only=True, monitor='val_loss', mode='min', save_weights_only=True)
    history = final_model.fit(X_train_1, tau_1_label, validation_split=0.2, epochs=500, batch_size=32688, shuffle=True, verbose=0, callbacks=[mcp_save])

    #plot_loss(history, "loss", "val_loss", "mse", "val_mse")
    
    # conv
    
    final_model = get_slearner_criteo_model()
    final_model.load_weights('../model_file/uplift/criteo/final_model/slearner/c_slearner_criteo_model_1.h5')
    T0 = np.zeros(shape=T_train.shape)
    T1 = np.ones(shape=T_train.shape)
    p0 = final_model.predict([X_train,  T0])
    p1 = final_model.predict([X_train,  T1])
    
    X_train_0 = X_train[(T_train.flatten() < 0.5), :]
    X_train_1 = X_train[(T_train.flatten() >= 0.5), :]

    X_train_0_p1_pre = p1[(T_train < 0.5)]
    X_train_1_p0_pre = p0[(T_train >= 0.5)]

    Y_conv_train_0 = Y_conv_train[(T_train < 0.5)]
    Y_conv_train_1 = Y_conv_train[(T_train >= 0.5)]

    tau_0_label = X_train_0_p1_pre - Y_conv_train_0
    tau_1_label = Y_conv_train_1 - X_train_1_p0_pre
    
    
    final_model = get_xlearner_criteo_model()
    final_model.compile(loss='mse', optimizer='adam', metrics=['mse'])

    mcp_save = ModelCheckpoint('../model_file/uplift/criteo/final_model/xlearner/c_conv_xlearner_criteo_model_tau_0_{}.h5'.format(sample), save_best_only=True, monitor='val_loss', mode='min', save_weights_only=True)
    history = final_model.fit(X_train_0, tau_0_label, validation_split=0.2, epochs=500, batch_size=32688, shuffle=True, verbose=0, callbacks=[mcp_save])

    #plot_loss(history, "loss", "val_loss", "mse", "val_mse")
    
    
    final_model = get_xlearner_criteo_model()
    final_model.compile(loss='mse', optimizer='adam', metrics=['mse'])

    mcp_save = ModelCheckpoint('../model_file/uplift/criteo/final_model/xlearner/c_conv_xlearner_criteo_model_tau_1_{}.h5'.format(sample), save_best_only=True, monitor='val_loss', mode='min', save_weights_only=True)
    history = final_model.fit(X_train_1, tau_1_label, validation_split=0.2, epochs=500, batch_size=32688, shuffle=True, verbose=0, callbacks=[mcp_save])

    
    # predict
    import sklearn 
    import sklearn.metrics
    from metric.Metric import *
    import keras
    import keras.backend as K
    import tensorflow as tf
    from keras.callbacks import LearningRateScheduler
    from keras.callbacks import ReduceLROnPlateau
    from model.roi_model import *
    
    # conv
    final_model = get_xlearner_criteo_model()
    final_model.load_weights('../model_file/uplift/criteo/final_model/xlearner/c_conv_xlearner_criteo_model_tau_0_{}.h5'.format(sample))

    tau_0_pre = final_model.predict(X_test)

    final_model = get_xlearner_criteo_model()
    final_model.load_weights('../model_file/uplift/criteo/final_model/xlearner/c_conv_xlearner_criteo_model_tau_1_{}.h5'.format(sample))

    tau_1_pre = final_model.predict(X_test)

    ex = 0.85
    
    xlearner_conv_pre = ex * tau_0_pre + (1 - ex) * tau_1_pre

#     xlearner_causalml_auuc = get_causalml_auuc(Y=Y_visit_test, T=T_test, ite_pred=xlearner_tau_pre)
    
#     xlearner_causalml_auuc_list.append(xlearner_causalml_auuc)
    # visit
    final_model = get_xlearner_criteo_model()
    final_model.load_weights('../model_file/uplift/criteo/final_model/xlearner/c_visit_xlearner_criteo_model_tau_0_{}.h5'.format(sample))

    tau_0_pre = final_model.predict(X_test)

    final_model = get_xlearner_criteo_model()
    final_model.load_weights('../model_file/uplift/criteo/final_model/xlearner/c_visit_xlearner_criteo_model_tau_1_{}.h5'.format(sample))

    tau_1_pre = final_model.predict(X_test)

    ex = 0.85
    
    xlearner_visit_pre = ex * tau_0_pre + (1 - ex) * tau_1_pre
    
    
    # roi
    
    roi_rank_pre_test = xlearner_conv_pre / np.where(abs(xlearner_visit_pre) < 1e-6, 1e-6, xlearner_visit_pre)
    
    roi_rank_pre_test_list.append(roi_rank_pre_test)
    DRP_aucc = get_uplift_model_aucc_no_show(t=(T_test > 0.5).flatten(), y_reward=Y_conv_test.flatten(), y_cost=Y_visit_test.flatten(), roi_pred=roi_rank_pre_test.flatten(), quantile=200)
    DRP_aucc_test_list.append(DRP_aucc)
    result_list.append(DRP_aucc[0])
    

print(result_list)
print(np.mean(result_list))
print(np.std(result_list))

# store test aucc for pic 
import pandas as pd

def get_aucc_cost_curve(aucc_list):
    delta_cost_list_group = np.array([aucc[1] for aucc in aucc_list])
    delta_reward_list_group = np.array([aucc[2] for aucc in aucc_list])
    
    avg_delta_cost_list = np.mean(delta_cost_list_group, axis=0)
    avg_delta_reward_list = np.mean(delta_reward_list_group, axis=0)
    
    df_aucc_cost_curve = pd.DataFrame(avg_delta_cost_list, columns=['delta_cost'])
    df_aucc_cost_curve['delta_reward'] = avg_delta_reward_list
    
    return df_aucc_cost_curve

average_list = get_aucc_cost_curve(DRP_aucc_test_list)
print("avg-aucc = ", np.sum(average_list['delta_reward'].values) / (average_list['delta_reward'].values[-1] * 201))
average_list.to_csv("../figure/sigir/{}.csv".format(model_name))

sample_value =  0.11
33639/33639 [==============================] - 100s 3ms/step


2024-01-26 10:35:50.297671: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x7fc0b48fdbe0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-01-26 10:35:50.297703: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2024-01-26 10:35:50.305197: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-01-26 10:35:50.354959: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:442] Loaded cuDNN version 8902
2024-01-26 10:35:50.500146: I ./tensorflow/compiler/jit/device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


14417/14417 [==============================] - 50s 3ms/step
AUCC =  0.5267306841045144
sample_value =  0.12
15728/15728 [==============================] - 46s 3ms/step
AUCC =  0.6040534093645222
sample_value =  0.13
17038/17038 [==============================] - 56s 3ms/step
AUCC =  0.6045817829233189
sample_value =  0.13999999999999999
18349/18349 [==============================] - 60s 3ms/step
AUCC =  0.5694313963407788
sample_value =  0.14999999999999997
19659/19659 [==============================] - 34s 2ms/step
AUCC =  0.6688038679583677
sample_value =  0.15999999999999998
20970/20970 [==============================] - 29s 1ms/step
AUCC =  0.6465664447617743
sample_value =  0.16999999999999998
22280/22280 [==============================] - 42s 2ms/step
AUCC =  0.6058516083959948
sample_value =  0.17999999999999997
23444/58103 [===========>..................] - ETA: 56s

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



24902/24902 [==============================] - 31s 1ms/step
AUCC =  0.45798614436961016
sample_value =  0.19999999999999996
49314/61161 [=======================>......] - ETA: 16s

...

 9543/26212 [=========>....................] - ETA: 20s

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



26212/26212 [==============================] - 31s 1ms/step
AUCC =  0.486587957085787
sample_value =  0.20999999999999996
36808/64219 [================>.............] - ETA: 36s

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



27523/27523 [==============================] - 32s 1ms/step
AUCC =  0.5732472116856787
[0.5267306841045144, 0.6040534093645222, 0.6045817829233189, 0.5694313963407788, 0.6688038679583677, 0.6465664447617743, 0.6058516083959948, 0.505690421477279, 0.45798614436961016, 0.486587957085787, 0.5732472116856787]
0.5681391753152387
0.06378294421173365
avg-aucc =  0.5654199985380989
CPU times: user 4h 48min 16s, sys: 31min 12s, total: 5h 19min 29s
Wall time: 3h 19min 47s


In [7]:
formatted_numbers = ["{:.4f}".format(number) for number in result_list]
output = " & ".join(formatted_numbers)

print(output)

0.5267 & 0.6041 & 0.6046 & 0.5694 & 0.6688 & 0.6466 & 0.6059 & 0.5057 & 0.4580 & 0.4866 & 0.5732
